# DSPy Programming - Signatures and Modules

In [1]:
# Install libraries
!pip install -q dspy openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 331.0/331.0 kB 22.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 146.5/146.5 kB 11.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.0/48.0 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 15.3/15.3 MB 84.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.1/278.1 kB 18.3 MB/s eta 0:00:00


In [9]:
# Load API key from Colab Secrets
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Configure the LM

In [11]:
# Configure the LM
import dspy

lm = dspy.LM("openai/gpt-4o-mini")
dspy.configure(lm=lm)

In [12]:
# Quick check: send a tiny request to confirm the model responds
test_response = lm("Say hello in one word.")
print(test_response)

['Hello!']


## Use DSPy built-in Module to Build a Sentiment Classifier

### Create a Signature

In [13]:
class SentimentClassifier(dspy.Signature):
  """Classify the sentiment of a text."""

  text: str = dspy.InputField(desc="input text to classify sentiment")
  sentiment: int = dspy.OutputField(desc="sentiment, the higher the more positive", ge=0, le=10)

In [14]:
str_signature = dspy.make_signature("text -> sentiment")

### Create a Module to Interact with the LM via the Signature

In [15]:
predict = dspy.Predict(SentimentClassifier)

In [16]:
output = predict(text="I am feeling pretty happy!")
print(output)

Prediction(
    sentiment=8
)


In [17]:
print(f"The sentiment is: {output.sentiment}")
print(f"The sentiment is: {output['sentiment']}")

The sentiment is: 8
The sentiment is: 8


In [18]:
# Try a different model
dspy.configure(lm=dspy.LM("openai/gpt-4o"))
print(predict(text="I am feeling pretty happy!"))

Prediction(
    sentiment=8
)


In [19]:
dspy.configure(lm=dspy.LM("openai/gpt-4o-mini"))

### Wait, Where is My Prompt?

In [20]:
# Inspect the last prompt
dspy.inspect_history(n=1)





[2026-06-07T10:13:45.550813]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeling pretty happy!

Respond with the corresponding output fields, starting with the field `[[ ## sentiment ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## sentiment ## ]]
8

[[ ## completed ## ]]







### Try a Different Built-in Module

In [21]:
cot = dspy.ChainOfThought(SentimentClassifier)

output = cot(text="I am feeling pretty happy!")
print(output)

Prediction(
    reasoning='The text expresses a positive emotion with the phrase "feeling pretty happy," indicating a high level of happiness. The use of the word "pretty" suggests a strong feeling rather than an overwhelming one. Therefore, the sentiment is positive but not maximally so.',
    sentiment=8
)


In [22]:
dspy.inspect_history(n=1)





[2026-06-07T10:15:51.727461]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

[[ ## text ## ]]
{text}

[[ ## reasoning ## ]]
{reasoning}

[[ ## sentiment ## ]]
{sentiment}        # note: the value you produce must be a single int value

[[ ## completed ## ]]
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeling pretty happy!

Respond with the corresponding output fields, starting with the field `[[ ## reasoning ## ]]`, then `[[ ## sentiment ## ]]` (must be formatted as a valid Python int), and then ending with the marker for `[[ ## completed ## ]]`.


Response:

[[ ## reasoning ## ]]
Th

### Use a Different Adapter

In [23]:
dspy.configure(adapter=dspy.JSONAdapter())

In [24]:
print(cot(text="I am feeling pretty happy!"))
dspy.inspect_history(n=1)

Prediction(
    reasoning="The text expresses a positive emotion ('happy'), indicating a high level of satisfaction or joy.",
    sentiment=8
)




[2026-06-07T10:16:36.151176]

System message:

Your input fields are:
1. `text` (str): input text to classify sentiment
Your output fields are:
1. `reasoning` (str): 
2. `sentiment` (int): sentiment, the higher the more positive
Constraints: greater than or equal to: 0, less than or equal to: 10
All interactions will be structured in the following way, with the appropriate values filled in.

Inputs will have the following structure:

[[ ## text ## ]]
{text}

Outputs will be a JSON object with the following fields.

{
  "reasoning": "{reasoning}",
  "sentiment": "{sentiment}        # note: the value you produce must be a single int value"
}
In adhering to this structure, your objective is: 
        Classify the sentiment of a text.


User message:

[[ ## text ## ]]
I am feeling pretty happy!

Respond with a JSON object in the following order

## Build a Program with Custom Module

In [25]:
class QuestionGenerator(dspy.Signature):
  """Generate a yes or no question in order to guess the celebrity name in users' mind. You can ask in general or directly guess the name if you think the signal is enough. You should never ask the same question in the past_questions."""
  past_questions: list[str] = dspy.InputField(desc="past questions asked")
  past_answers: list[bool] = dspy.InputField(desc="past answers")
  new_question: str = dspy.OutputField(desc="new question that can help narrow down the celebrity name")
  guess_made: bool = dspy.OutputField(desc="If the new_question is the celebrity name guess, set to True, if it is still a general question set to False")


class Reflection(dspy.Signature):
  """Provide reflection on the guessing process"""
  correct_celebrity_name: str = dspy.InputField(desc="the celebrity name in user's mind")
  final_guessor_question: str = dspy.InputField(desc="the final guess or question LM made")
  past_questions: list[str] = dspy.InputField(desc="past questions asked")
  past_answers: list[bool] = dspy.InputField(desc="past answers")

  reflection: str = dspy.OutputField(desc="reflection on the guessing process, including what was done well and what can be improved")

def ask(prompt, valid_responses=("y", "n")):
  while True:
    response = input(f"{prompt} ({'/'.join(valid_responses)}): ").strip().lower()
    if response in valid_responses:
      return response
    print(f"Please enter one of: {', '.join(valid_responses)}")

class CelebrityGuess(dspy.Module):
  def __init__(self, max_tries=10):
    super().__init__()

    self.question_generator = dspy.ChainOfThought(QuestionGenerator)
    self.reflection = dspy.ChainOfThought(Reflection)

    self.max_tries = 20

  def forward(self):
    celebrity_name = input("Please think of a celebrity name, once you are ready, type the name and press enter...")
    past_questions = []
    past_answers = []

    correct_guess = False

    for i in range(self.max_tries):
      question = self.question_generator(past_questions=past_questions, past_answers=past_answers)
      answer = ask(f"{question.new_question}").lower() == "y"
      past_questions.append(question.new_question)
      past_answers.append(answer)

      if question.guess_made and answer:
        correct_guess = True
        break

    if correct_guess:
      print("Yay! I got it right!")
    else:
      print("Oops, I couldn't guess it right.")

    reflection = self.reflection(
        correct_celebrity_name=celebrity_name,
        final_guessor_question=question.new_question,
        past_questions=past_questions,
        past_answers=past_answers,
    )
    print(reflection.reflection)

In [26]:
celebrity_guess = CelebrityGuess()

In [28]:
celebrity_guess()

Please think of a celebrity name, once you are ready, type the name and press enter...Lionel Messi
Is this celebrity primarily known for their work in film? (y/n): n
Is this celebrity known for their work in television? (y/n): n
Is this celebrity primarily known for their work in music? (y/n): n
Is this celebrity primarily known for their achievements in sports? (y/n): y
Is this celebrity a professional athlete? (y/n): y
Does this athlete primarily compete in a team sport? (y/n): y
Does this athlete play in a major league in the United States? (y/n): y
Does this athlete play professional basketball? (y/n): n
Does this athlete play professional football? (y/n): y
Does this athlete primarily play baseball? (y/n): n
Does this athlete primarily play baseball? (y/n): n
Does this athlete primarily play professional hockey? (y/n): n
Does this athlete play professional baseball? (y/n): n
Does this athlete play professional soccer? (y/n): y
Is this athlete a player in Major League Baseball (MLB

In [29]:
celebrity_guess()

Please think of a celebrity name, once you are ready, type the name and press enter...Lebron James
Is this celebrity primarily known for their work in film? (y/n): n
Is this celebrity known for their work in television? (y/n): n
Is this celebrity primarily known for their work in music? (y/n): n
Is this celebrity primarily known for their achievements in sports? (y/n): y
Is this celebrity a professional athlete? (y/n): y
Does this athlete primarily compete in a team sport? (y/n): y
Does this athlete play in a major league in the United States? (y/n): y
Does this athlete play professional basketball? (y/n): y
Does this athlete currently play for the Los Angeles Lakers? (y/n): y
Is this celebrity LeBron James? (y/n): y
Yay! I got it right!
The guessing process was largely effective due to the structured approach in forming relevant questions that led to a logical elimination of other possibilities. Starting from broader categories like film, television, and music allowed for a quick dism

## Save and Load

In [31]:
celebrity_guess.save("celebrity.json", save_program=False)

In [32]:
celebrity_guess.load("celebrity.json")

In [33]:
celebrity_guess.save("celebrity/", save_program=True)

2026/06/07 10:31:26 WARNING dspy.primitives.base_module: Loading untrusted .pkl files can run arbitrary code, which may be dangerous. To avoid this, prefer saving using json format using module.save("module.json").


In [39]:
loaded = dspy.load("celebrity/", allow_pickle=True)

In [40]:
loaded()

Please think of a celebrity name, once you are ready, type the name and press enter...Cristiano Ronaldo
Is this celebrity primarily known for their work in film? (y/n): n
Is this celebrity known for their work in television? (y/n): n
Is this celebrity primarily known for their work in music? (y/n): n
Is this celebrity primarily known for their achievements in sports? (y/n): y
Is this celebrity a professional athlete? (y/n): y
Does this athlete primarily compete in a team sport? (y/n): y
Does this athlete play in a major league in the United States? (y/n): n
Is this athlete currently active in their sport? (y/n): y
Does this athlete primarily compete in an international league or tournament not based in the United States? (y/n): n
Is this athlete from a European country? (y/n): y
Is this athlete primarily known for their work in soccer? (y/n): y
Is this athlete from England? (y/n): n
Is this athlete from France? (y/n): n
Is this athlete from Spain? (y/n): n
Is this athlete Lionel Messi?